In [2]:
#carregando e ordenando dados
import pandas as pd 
import numpy as np

base = pd.read_parquet("../data/processed/dataset_modelagem.parquet")
base = base.sort_values("ini").reset_index(drop=True)
print(base.shape)



(4055, 65)


In [3]:
#entradas x e y
X_COLS = [
    "comprimento", "qtd_sessao", "tempo_prog_sessao_min", "quantidade_ordem",
    "troca_condutor", "troca_terminal_a", "troca_terminal_b", "troca_cor", "troca_ma_a", "troca_ma_b",
    "mesmo_sub_item", "mesma_programacao", "ocioso_antes_min", "lacuna_antes_6h",
    "hora_ini", "dia_semana", "ordem_no_dia", "n_subitens_kit",
    "n_ocorrencias_previas_sub", "dur_media_hist_sub", "razao_media_hist_sub",
    "setup_pcp", "maquina_diferente_programada",
]
base["mc_68"] = (base["mc"] == "68").astype(int)

X = base[X_COLS + ["mc_68"]].astype(float)
y = base["y_abastecimento"]

In [ ]:
#treino e teste

corte = int(len(base) * 0.8)
X_tr, X_te = X.iloc[:corte], X.iloc[corte:]
y_tr, y_te = y.iloc[:corte], y.iloc[corte:]

print("Treino:", base["ini"].iloc[0].date(), "a", base["ini"].iloc[corte - 1].date(), f"({corte} sessões)")
print("Teste: ", base["ini"].iloc[corte].date(), "a", base["ini"].iloc[-1].date(), f"({len(base) - corte} sessões)")
print("Taxa de abastecimento – treino:", round(y_tr.mean(), 3), "| teste:", round(y_te.mean(), 3))

Treino: 2026-04-22 a 2026-08-31 (3244 sessões)
Teste:  2026-08-31 a 2026-10-07 (811 sessões)
Taxa de abastecimento – treino: 0.72 | teste: 0.736


In [ ]:
#regressão 
#colocar isso aqui no artigo, importante montar a tabela de regressão com acuracia, recall, f1, auc

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def avalia(nome, modelo):
    modelo.fit(X_tr, y_tr)
    prob = modelo.predict_proba(X_te)[:, 1]
    pred = (prob >= 0.5).astype(int)
    return {"modelo": nome,
            "acuracia": accuracy_score(y_te, pred),
            "precisao": precision_score(y_te, pred, zero_division=0),
            "recall":   recall_score(y_te, pred),
            "F1":       f1_score(y_te, pred),
            "AUC":      roc_auc_score(y_te, prob)}

baseline = DummyClassifier(strategy="most_frequent")
logreg = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000))

resultados = pd.DataFrame([avalia("Baseline (sempre 'sim')", baseline),
                           avalia("Regressão Logística", logreg)])
resultados.round(3)

,modelo,acuracia,precisao,recall,F1,AUC
0,Baseline (sempre 'sim'),0.736,0.736,1.000,0.848,0.500
1,Regressão Logística,0.800,0.836,0.906,0.870,0.849


In [ ]:
#matriz de confusão, também é importante colocar no artigo, contribui para a precisão do modelo.

from sklearn.metrics import confusion_matrix

prob = logreg.predict_proba(X_te)[:, 1]
print(confusion_matrix(y_te, (prob >= 0.5).astype(int)))

pesos = pd.Series(logreg[-1].coef_[0], index=X.columns).sort_values()
pesos.round(2)

[[108 106]
 [ 56 541]]


quantidade_ordem               -0.55
troca_condutor                 -0.38
ordem_no_dia                   -0.21
lacuna_antes_6h                -0.19
qtd_sessao                     -0.09
razao_media_hist_sub           -0.05
mc_68                          -0.03
maquina_diferente_programada   -0.02
dia_semana                     -0.01
troca_terminal_b                0.01
n_subitens_kit                  0.01
troca_ma_b                      0.02
mesmo_sub_item                  0.03
troca_terminal_a                0.04
mesma_programacao               0.05
hora_ini                        0.05
troca_ma_a                      0.06
setup_pcp                       0.10
ocioso_antes_min                0.12
dur_media_hist_sub              0.13
troca_cor                       0.19
n_ocorrencias_previas_sub       0.19
comprimento                     0.27
tempo_prog_sessao_min           2.18
dtype: float64